# Create individual json refs for FVCOM Hindcast files on AWS
This workflow was a struggle. I needed to do it in pieces using different code and environments
* The first script (this one) creates individual jsons using the very latest version of kerchunk (actually a PR) with a zarr3 environment.  It was not possible to do any subchunking -- if I tried, I got a "loop not running" error.
* Another script using a zarr=2 environment, with `kerchunk=0.2.7` then combined the JSON into a single set of parquet references
* Another script converted the individual non-subchunked jsons into individual subchunked jsons.
* I could not combine these yet, as I haven't found enough memory. 

In [ ]:
import fsspec
import xarray as xr
from kerchunk.netCDF3 import NetCDF3ToZarr
import kerchunk.combine
import kerchunk.zarr
from kerchunk.combine import MultiZarrToZarr
from fsspec.implementations.reference import LazyReferenceMapper
import os
from kerchunk.utils import subchunk, inline_array

In [ ]:
from dotenv import load_dotenv

In [ ]:
load_dotenv('/shared/users/nebari-setup/chen_keys.env') 

In [ ]:
so = dict(anon=False, skip_instance_cache=True, use_listings_cache=False,
                           config_kwargs={'connect_timeout':5, 'read_timeout':5}
         )

In [ ]:
fs = fsspec.filesystem('s3', **so)

In [ ]:
flist = sorted(fs.glob('umassd-fvcom/gom3/hindcast/*.nc'))

In [ ]:
print(len(flist), 'files')
print(flist[0])
print(flist[-1])

In [ ]:
identical_dims = ['partition',
 'x',
 'y',
 'lon',
 'lat',
 'xc',
 'yc',
 'lonc',
 'latc',
 'siglay',
 'siglev',
 'h',
 'nv',
 'nbe',
 'ntsn',
 'nbsn',
 'ntve',
 'nbve',
 'a1u',
 'a2u',
 'aw0',
 'awx',
 'awy',
 'art2',
 'art1',
 'nprocs']

## Create Dask cluster with credentials for AWS open data bucket

In [ ]:
from pathlib import Path
import numpy as np
import ujson

In [ ]:
json_dir = 's3://umassd-fvcom/gom3/hindcast/individual_jsons3'

In [ ]:
flist[0]

In [ ]:
def gen_json(u):
    with fs.open(u, **so) as infile:
        fname = Path(u).stem
        d0 = NetCDF3ToZarr(f's3://{u}', storage_options=so,
                  inline_threshold=400, version=2).translate()
        # for v in siglev_vars:
        #     d0 = subchunk(store=d0, variable=v, factor=nlev)
        # for v in siglay_vars:
        #     d0 = subchunk(store=d0, variable=v, factor=nlay)
        outf = f'{json_dir}/{fname}.json'
        with fs.open(outf, 'wb') as f:
            f.write(ujson.dumps(d0).encode());
    return outf

In [ ]:
%%time
import os
cluster_type = 'Gateway'

if cluster_type == 'Gateway':
    from dask_gateway import Gateway

    gateway = Gateway()  # instantiate Dask gateway 

    # Cluster options on Nebari 
    options = gateway.cluster_options()
    options.conda_environment='global/global-pangeo-dev'  # comment out for Daskhub or Planetary Computer
    options.profile = 'Small Worker'   # comment out for Daskhub or Planetary Computer
    options.environment_vars = {'AWS_ACCESS_KEY_ID':os.environ['AWS_ACCESS_KEY_ID'],
                                'AWS_SECRET_ACCESS_KEY':os.environ['AWS_SECRET_ACCESS_KEY']}
    # Create a Dask Gateway cluster
    cluster = gateway.new_cluster(options)

    # Get the Dask client for the Dask Gateway cluster
    client = cluster.get_client()

    # Scale the cluster
    cluster.adapt(minimum=4, maximum=30)

In [ ]:
client

In [ ]:
worker_max=30

In [ ]:
%%time
import dask
_ = dask.compute(*[dask.delayed(gen_json)(f) for f in flist], retries=20)

In [ ]:
#client.close()

In [ ]:
#cluster.shutdown()